# Feature Engineering – Industrial Chiller Digital Twin

Este notebook realiza a etapa de engenharia de atributos do projeto.

A partir dos dados brutos gerados na camada `raw`, são criadas variáveis derivadas
com significado físico, como diferença de temperatura, vazão mássica, carga térmica
removida e coeficiente de performance (COP).

Essa etapa conecta os dados operacionais à interpretação termodinâmica do sistema.

In [0]:
import pandas as pd
import numpy as np
from pathlib import Path

LAKEHOUSE_PATH = "/Volumes/analytics/digital_twin/data"

# LOCALMENTE
# LAKEHOUSE_PATH = "../data"

RAW_PATH = f"{LAKEHOUSE_PATH}/raw"
SILVER_PATH = f"{LAKEHOUSE_PATH}/silver"

input_file = f"{RAW_PATH}/dados_refrigeracao.csv"
output_file = f"{SILVER_PATH}/dados_refrigeracao_enriched.csv"

df = pd.read_csv(input_file)
df.head()

## Dicionário de Variáveis

As variáveis utilizadas representam medições típicas de um sistema de refrigeração industrial:

- **temp_entrada_c**  
  Temperatura da água na entrada do sistema (°C).

- **temp_saida_c**  
  Temperatura da água após o processo de resfriamento (°C).

- **vazao_m3_s**  
  Vazão volumétrica de água circulante no sistema (m³/s).

- **potencia_kw**  
  Potência elétrica total consumida pelo sistema (kW).

- **pressao_oleo_bar**  
  Pressão do óleo do compressor (bar).

- **nivel_tanque_pct**  
  Nível do tanque do sistema (%).

Essas variáveis permitem estimar a carga térmica removida e avaliar o desempenho energético por meio do COP.


In [0]:
# timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# remover linhas sem timestamp válido
df = df.dropna(subset=["timestamp"]).copy()

# ordenar cronologicamente
df = df.sort_values("timestamp").reset_index(drop=True)

# Remove timestamps duplicados (mantém o último valor)
df = df.drop_duplicates(subset=["timestamp"], keep="last").copy()

In [0]:
df["flag_temp_invertida"] = df["temp_entrada_c"] <= df["temp_saida_c"]
df["flag_vazao_invalida"] = df["vazao_m3_h"] <= 0
df["flag_potencia_invalida"] = df["potencia_kw"] <= 0

In [0]:
RHO = 1000.0          # kg/m³
CP = 4.186            # kJ/(kg·°C)

# diferença de temperatura
df["dT_c"] = df["temp_entrada_c"] - df["temp_saida_c"]

# vazão volumétrica em m³/s
df["vazao_m3_s"] = df["vazao_m3_h"] / 3600

# vazão mássica em kg/s
df["m_dot_kg_s"] = df["vazao_m3_s"] * RHO

# carga térmica removida em kW
# como CP está em kJ/(kg·°C), o resultado já sai em kW
df["qdot_kw"] = df["m_dot_kg_s"] * CP * df["dT_c"]

# COP
df["cop"] = df["qdot_kw"] / df["potencia_kw"]

In [0]:
df["flag_dT_invalido"] = df["dT_c"] <= 0
df["flag_cop_invalido"] = (df["cop"] <= 0) | (df["cop"] > 20)

df["flag_invalido"] = (
    df["flag_temp_invertida"] |
    df["flag_vazao_invalida"] |
    df["flag_potencia_invalida"] |
    df["flag_dT_invalido"] |
    df["flag_cop_invalido"]
)

In [0]:
display(df)

In [0]:
Path(SILVER_PATH).mkdir(parents=True, exist_ok=True)
df.to_csv(output_file, index=False)

print("Arquivo salvo em:", output_file)